# CS781 Capstone -- Phase 3: Rhetorical Role Prediction

## Stage 2: Shared setup and data

This notebook is built in stages, as agreed. This stage contains everything every one of the
8 experiments will reuse: the four Phase 1 backbones, the shared tokenizer, the Phase 3
data pipeline, the metrics / dev-calibration utilities, and the submission writer.

**No experiment-specific code (LoRA, LoRMA, IA3, Prompt/Prefix Tuning, KD, ensembling) is in
this stage.** Those come next, once you've confirmed this stage runs cleanly on your data.



## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Install dependencies

`torchao` is uninstalled first as a known Kaggle/peft version conflict -- harmless if it was
never installed.

In [ ]:
!pip uninstall -y torchao -q 2>/dev/null
!pip -q install regex peft accelerate
import peft
print("peft version:", peft.__version__)

## 4. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil
import numpy as np
import torch

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "CL": "Phase_3_RR/Train/CL_train.json",
    "IT": "Phase_3_RR/Train/IT_train.json",
}
DEV_FILES = {
    "CL": "Phase_3_RR/Train/CL_dev.json",
    "IT": "Phase_3_RR/Train/IT_dev.json",
}
TEST_FILES = {
    "CL": "Phase_3_RR/Test/CL_test.json",
    "IT": "Phase_3_RR/Test/IT_test.json",
}
LABEL_VOCAB_FILE = "Phase_3_RR/Train/label_vocab.json"

# ---------------- Phase 1 checkpoints (all share the same tokenizer) ----------------
MODEL_PATHS = {
    "gpt2":    "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":  "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":     "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm":"/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD = 768, 12, 12
N_KV_HEAD = 4   # GQA and Prefix-LM only

# MAX_SENT_LEN is set later, from the actual token-length distribution (Section 7 below)
MAX_SENT_LEN = None

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase3"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    """Every experiment gets its own folder: model, run_info.json, submission.zip."""
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
        "loss_plot_path": os.path.join(d, "training_validation_loss.png"),
    }

print("Configuration loaded.")

## 5. Tokenizer

Same from-scratch byte-level BPE used for all four Phase 1 models -- loaded once from the GQA checkpoint and reused everywhere, since it's the same file across all four model folders.

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))

# sanity: confirm all four checkpoints really do ship the identical tokenizer
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer vocab size differs!"
print("All four checkpoints share the same tokenizer vocab size.")

## 6. Load Phase 3 data

In [ ]:
from huggingface_hub import hf_hub_download

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

train_docs, dev_docs, test_docs = {}, {}, {}
for domain, path in TRAIN_FILES.items():
    train_docs[domain] = hf_json(path)
for domain, path in DEV_FILES.items():
    dev_docs[domain] = hf_json(path)
for domain, path in TEST_FILES.items():
    test_docs[domain] = hf_json(path)

label_vocab = hf_json(LABEL_VOCAB_FILE)
print({d: len(v) for d, v in train_docs.items()}, "train docs")
print({d: len(v) for d, v in dev_docs.items()}, "dev docs")
print({d: len(v) for d, v in test_docs.items()}, "test docs")
print("label_vocab:", label_vocab)

## 7. Label vocabulary

13 labels, `"None"` included as a real string label, loaded from `label_vocab.json` rather than hardcoded, so a mismatch in spelling/casing surfaces immediately.

In [ ]:
# label_vocab.json's exact shape isn't guaranteed (list vs dict) -- handle both.
if isinstance(label_vocab, dict):
    LABELS = list(label_vocab.keys()) if all(isinstance(v, (int, float)) for v in label_vocab.values()) else list(label_vocab.values())
else:
    LABELS = list(label_vocab)

assert len(LABELS) == 13, f"expected 13 labels, got {len(LABELS)}: {LABELS}"
assert "None" in LABELS, "literal label 'None' missing from label_vocab"

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(LABELS)

## 8. Flatten documents into sentence-level records

Keeps `doc_id`, `domain`, and `sent_idx` on every record so predictions can be reassembled in the exact original order per document later.

In [ ]:
def flatten(docs_by_domain, with_labels=True):
    records = []
    for domain, docs in docs_by_domain.items():
        if isinstance(docs, dict):
            items = list(docs.items())          # test: {doc_id: [sentences]}
        else:
            items = [(d["id"], d) for d in docs]  # train/dev: [{"id", "text", "labels"}, ...]

        for doc_id, doc in items:
            sents = doc if isinstance(doc, list) else doc["text"]
            labels = doc.get("labels") if (with_labels and isinstance(doc, dict)) else None
            if with_labels and labels is not None:
                assert len(sents) == len(labels), (doc_id, len(sents), len(labels))
            for i, s in enumerate(sents):
                rec = {"doc_id": doc_id, "domain": domain, "sent_idx": i, "text": s}
                if with_labels and labels is not None:
                    rec["label"] = labels[i]
                records.append(rec)
    return records

train_records = flatten(train_docs, with_labels=True)
dev_records   = flatten(dev_docs,   with_labels=True)
test_records  = flatten(test_docs,  with_labels=False)

print("train sentences:", len(train_records))
print("dev sentences:", len(dev_records))
print("test sentences:", len(test_records))

bad_labels = {r["label"] for r in train_records + dev_records} - set(LABELS)
assert not bad_labels, f"labels not in vocab: {bad_labels}"
print("all labels valid")

## 8b. Position and domain features per sentence

Rhetorical roles depend strongly on where a sentence sits in the document (facts early, rulings late). Each record gets four
features: relative position, log distance from the start, log distance to the end, and a domain flag (IT = 1).

In [ ]:
from collections import Counter

def add_features(records):
    n_sent = Counter((r["domain"], r["doc_id"]) for r in records)
    for r in records:
        L, i = n_sent[(r["domain"], r["doc_id"])], r["sent_idx"]
        r["feats"] = [i / max(L - 1, 1),
                      math.log1p(i) / 6,
                      math.log1p(L - 1 - i) / 6,
                      1.0 if r["domain"] == "IT" else 0.0]

for _recs in (train_records, dev_records, test_records):
    add_features(_recs)
print("example feats:", train_records[0]["feats"], train_records[5]["feats"])


## 9. Sentence token-length distribution -> choose MAX_SENT_LEN

Data-driven, not assumed. Covers roughly the 99th percentile, rounded up to a clean multiple of 16.

In [ ]:
lengths = np.array([len(tokenizer.encode(r["text"])) for r in train_records])
percentiles = [50, 75, 90, 95, 99, 99.5, 100]
for p in percentiles:
    print(f"p{p:>5}: {np.percentile(lengths, p):.1f}")

p99 = np.percentile(lengths, 99)
MAX_SENT_LEN = int(math.ceil(p99 / 16) * 16)
MAX_SENT_LEN = min(MAX_SENT_LEN, MODEL_CONTEXT_LENGTH)
coverage = (lengths <= MAX_SENT_LEN).mean()
print(f"chosen MAX_SENT_LEN = {MAX_SENT_LEN}  (covers {coverage:.2%} of train sentences without truncation)")

## 10. Dataset / collator -- sentence-level input (experiments #1-#6)

Each sentence is encoded independently, padded/truncated to `MAX_SENT_LEN`. **New:** every item also carries four
document-position features (`feats`, built in Section 8b) so the classifier knows *where* in the document the sentence sits.
The neighbouring-sentence (context-aware) dataset for experiment #7 is still a separate collator, built later.

In [ ]:
PAD_ID = tokenizer.eos_token_id

class SentenceDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids = tokenizer.encode(r["text"])[:MAX_SENT_LEN]
        item = {"ids": ids, "idx": idx, "feats": r["feats"]}
        if self.with_labels:
            item["label"] = LABEL2ID[r["label"]]
        return item


def sentence_collator(batch):
    L = MAX_SENT_LEN
    B = len(batch)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, :n] = torch.tensor(item["ids"])
        attention_mask[i, :n] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask,
           "record_idx": torch.tensor([item["idx"] for item in batch]),
           "feats": torch.tensor([item["feats"] for item in batch], dtype=torch.float)}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([item["label"] for item in batch])
    return out

_test_batch = sentence_collator([SentenceDataset(train_records)[i] for i in range(4)])
print({k: v.shape for k, v in _test_batch.items()})

## 11. Class weights (joint CL+IT, inverse-frequency, clipped)

In [ ]:
from collections import Counter

train_label_counts = Counter(r["label"] for r in train_records)
total = sum(train_label_counts.values())
print(f"{'label':25s} {'count':>7s} {'%':>6s}")
for l in LABELS:
    c = train_label_counts.get(l, 0)
    print(f"{l:25s} {c:7d} {100*c/total:5.1f}%")

weights_list = []
for l in LABELS:
    c = train_label_counts.get(l, 0)
    weights_list.append(total / (len(LABELS) * c) if c > 0 else 0.0)
class_weights = torch.tensor(np.clip(weights_list, 0.1, 10.0), dtype=torch.float).to(device)
print("class weights:", class_weights.tolist())

## 12. Backbone architectures (all four Phase 1 models)

In [ ]:
import math as _m
import torch.nn as nn
import torch.nn.functional as F

# ---------------- shared causal attention block (GPT-2 baseline) ----------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- RMSNorm + SwiGLU (Model 2) ----------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)   # gate
        self.w3 = nn.Linear(n_embd, hidden, bias=False)   # up
        self.w2 = nn.Linear(hidden, n_embd, bias=False)   # down
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass   # identical mechanics; RMSNorm is applied outside, in the block


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- Grouped-Query Attention (Model 3 / shared with Model 4) ----------------
class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- Prefix-LM (Model 4, same block type as GQA) ----------------
class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

## 13. Backbone registry and loader

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,       "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,            "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_backbone(name):
    """Fresh instance of the named Phase 1 backbone, weights loaded, lm_head dropped
    (not needed for classification)."""
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    result = model.load_state_dict(state)
    print(f"[{name}] load_state_dict: {result}")
    if hasattr(model, "lm_head"):
        del model.lm_head
    return model

# smoke test: confirm every checkpoint actually loads cleanly before building anything on top
for name in BACKBONE_REGISTRY:
    m = load_backbone(name)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name}: {n_params/1e6:.1f}M params\n")
    del m

## 14. Sentence classifier wrapper

Wraps any backbone with **last-token + mean pooling** and a small MLP head. The head also receives the four
position/domain features (`feats`). `bidirectional=True` is only ever used for the separate Prefix-LM ablation;
every other experiment leaves it `False` (causal, matching Phase 1 pretraining).

In [ ]:
N_FEATS = 4   # relative position, log position from start, log position from end, domain flag (IT=1)

class SentenceClassifier(nn.Module):
    def __init__(self, lm, n_labels, n_embd, dropout=0.1, bidirectional=False, n_feats=N_FEATS):
        super().__init__()
        self.lm = lm
        self.bidirectional = bidirectional
        self.n_feats = n_feats
        self.dropout = nn.Dropout(dropout)
        # head input = last-token state + mean-pooled state + position/domain features
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd + n_feats, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, n_labels)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None,
                logit_offsets=None, feats=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        x = self.lm.drop(x)

        if self.bidirectional:
            attn_mask = torch.ones(B, 1, T, T, dtype=torch.bool, device=input_ids.device)
            if attention_mask is not None:
                attn_mask = attn_mask & attention_mask[:, None, None, :].bool()
        else:
            attn_mask = None   # falls back to each block's own causal buffer
            if attention_mask is not None:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).view(1, 1, T, T)
                attn_mask = causal & attention_mask[:, None, None, :].bool()

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        # pooling: last real (non-pad) token, and the mean over real tokens
        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            m = attention_mask.unsqueeze(-1).float()
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
            m = torch.ones(B, T, 1, device=input_ids.device)
        last = x[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(x.dtype)
        if feats is None:
            feats = torch.zeros(B, self.n_feats, device=input_ids.device)
        pooled = torch.cat([last, mean, feats.to(x.dtype)], dim=-1)

        logits = self.head(self.dropout(pooled))
        if logit_offsets is not None:
            logits = logits + logit_offsets

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        return {"loss": loss, "logits": logits}

print("SentenceClassifier defined.")


## 15. Metrics and dev-only logit-offset calibration

`macro_prf` computes macro-F1/P/R, with an optional domain filter for the CL/IT split the brief requires.
`calibrate_logit_offsets` does a small coordinate-ascent search over per-class additive offsets to raise dev macro-F1.
**New:** epochs are selected on the *uncalibrated* dev macro-F1, and `calibration_check` tests the offsets on held-out
dev documents (fit on one half, evaluate on the other) before they are used for test.

In [ ]:
def macro_prf(y_true, y_pred, n_labels=len(LABELS)):
    cnts = np.zeros((n_labels, 3))   # tp, pred, gold
    for t, p in zip(y_true, y_pred):
        cnts[p, 1] += 1
        cnts[t, 2] += 1
        if t == p:
            cnts[t, 0] += 1
    P = np.divide(cnts[:, 0], cnts[:, 1], out=np.zeros(n_labels), where=cnts[:, 1] > 0)
    R = np.divide(cnts[:, 0], cnts[:, 2], out=np.zeros(n_labels), where=cnts[:, 2] > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(n_labels), where=(P + R) > 0)
    return P, R, F


def print_scores(y_true, y_pred, domains=None, title=""):
    print(f"--- {title} ---")
    P, R, F = macro_prf(y_true, y_pred)
    print(f"{'label':25s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    for i, l in enumerate(LABELS):
        print(f"{l:25s} {P[i]:6.3f} {R[i]:6.3f} {F[i]:6.3f}")
    print(f"MACRO  P {P.mean():.4f}  R {R.mean():.4f}  F1 {F.mean():.4f}")
    if domains is not None:
        for dom in sorted(set(domains)):
            idx = [i for i, d in enumerate(domains) if d == dom]
            dP, dR, dF = macro_prf([y_true[i] for i in idx], [y_pred[i] for i in idx])
            print(f"[{dom}]  P {dP.mean():.4f}  R {dR.mean():.4f}  F1 {dF.mean():.4f}")
    return F.mean()


@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], feats=b.get("feats"))
        all_logits.append(out["logits"].float().cpu())
        if "labels" in b:
            all_labels.append(b["labels"].cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels) if all_labels else None
    return logits, labels


def calibrate_logit_offsets(logits, labels, n_labels=len(LABELS), n_rounds=3, step=0.5, n_steps=9, verbose=True):
    """Coordinate ascent: for each label in turn, try a small range of additive offsets and
    keep whichever improves dev macro-F1. Fit on dev only; the returned offsets are then frozen
    and applied identically at test time."""
    offsets = torch.zeros(n_labels)
    y_true = labels.numpy()
    candidates = np.linspace(-step * n_steps, step * n_steps, 2 * n_steps + 1)
    best_f1 = macro_prf(y_true, (logits + offsets).argmax(-1).numpy())[2].mean()
    for _ in range(n_rounds):
        improved = False
        for c in range(n_labels):
            base = offsets[c].item()
            best_c, best_c_f1 = base, best_f1
            for delta in candidates:
                offsets[c] = delta
                f1 = macro_prf(y_true, (logits + offsets).argmax(-1).numpy())[2].mean()
                if f1 > best_c_f1:
                    best_c, best_c_f1 = delta, f1
            offsets[c] = best_c
            if best_c_f1 > best_f1 + 1e-6:
                best_f1 = best_c_f1
                improved = True
        if not improved:
            break
    if verbose:
        print(f"calibration: dev macro-F1 {best_f1:.4f} (offsets: {[round(o,2) for o in offsets.tolist()]})")
    return offsets, best_f1

def raw_macro_f1(logits, labels):
    """Dev macro-F1 with NO offsets -- used to select epochs."""
    return macro_prf(labels.numpy(), logits.argmax(-1).numpy())[2].mean()


def calibration_check(logits, labels, doc_keys, seed=0):
    """Honest test of the offsets: fit them on one half of the dev DOCUMENTS and evaluate on the other half
    (both directions). Returns the mean change in macro-F1 on the held-out halves."""
    keys = np.array([f"{d}|{i}" for d, i in doc_keys])
    uniq = np.unique(keys)
    rng = np.random.RandomState(seed); rng.shuffle(uniq)
    A = np.isin(keys, uniq[: len(uniq) // 2]); B = ~A
    gains = []
    for fit_m, test_m in ((A, B), (B, A)):
        fit_t, test_t = torch.from_numpy(fit_m), torch.from_numpy(test_m)
        offs, _ = calibrate_logit_offsets(logits[fit_t], labels[fit_t], verbose=False)
        base = raw_macro_f1(logits[test_t], labels[test_t])
        cal = macro_prf(labels[test_t].numpy(), (logits[test_t] + offs).argmax(-1).numpy())[2].mean()
        gains.append(cal - base)
        print(f"  held-out half: macro-F1 without offsets {base:.4f} | with offsets {cal:.4f}")
    return float(np.mean(gains))

print("Metrics utilities defined.")

## 16. Submission writer

In [ ]:
def write_submission(paths, model, offsets=None):
    test_ds = SentenceDataset(test_records, with_labels=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=sentence_collator)

    model.eval()
    all_logits = []
    with torch.no_grad():
        for b in test_loader:
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], feats=b["feats"])
            all_logits.append(out["logits"].float().cpu())
    logits = torch.cat(all_logits)
    if offsets is not None:
        logits = logits + offsets.cpu()          # apply the dev-calibrated offsets
    preds = logits.argmax(-1).tolist()

    by_doc = {}
    for rec, p in zip(test_records, preds):
        by_doc.setdefault(rec["doc_id"], {})[rec["sent_idx"]] = ID2LABEL[p]

    prediction, expected_ids = {}, set()
    for domain, docs in test_docs.items():
        items = docs.items() if isinstance(docs, dict) else [(d["id"], d["text"]) for d in docs]
        for doc_id, sents in items:
            expected_ids.add(doc_id)
            prediction[doc_id] = [by_doc[doc_id][i] for i in range(len(sents))]

    assert set(prediction) == expected_ids

    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    print("saved:", paths["submission_zip"])
    return prediction

## 17. Generic training loop

Shared by every experiment; what differs per experiment is which parameters are passed to the optimizer (full fine-tune, frozen+adapter, etc.), supplied from outside.
Checkpoints are chosen on **uncalibrated** dev macro-F1. After the best checkpoint is restored, the offsets are checked on held-out dev documents and only used if they help.

In [ ]:
def train_classifier(model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
                     class_weights, epochs, patience, paths, scaler=None, min_cal_gain=0.005):
    scaler = scaler or torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = {"train": [], "dev_f1": []}          # dev_f1 = UNCALIBRATED dev macro-F1
    best_f1, bad_epochs = 0.0, 0
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(train_loader):
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**b, class_weights=class_weights)
            scaler.scale(out["loss"]).backward()
            run += out["loss"].item(); cnt += 1
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        logits, labels = get_logits(model, dev_loader)
        f1 = raw_macro_f1(logits, labels)           # epoch selection uses NO offsets
        history["train"].append(run / cnt); history["dev_f1"].append(f1)
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated) {f1:.4f} | {time.time()-t0:.0f}s")

        if f1 > best_f1:
            best_f1, bad_epochs = f1, 0
            torch.save(model.state_dict(), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
    logits, labels = get_logits(model, dev_loader)

    # offsets: check on held-out dev documents first, use them only if they help
    print("calibration check (fit on one half of the dev documents, test on the other half):")
    gain = calibration_check(logits, labels, [(r["domain"], r["doc_id"]) for r in dev_records])
    print(f"mean held-out gain from offsets: {gain:+.4f} (threshold {min_cal_gain})")
    if gain > min_cal_gain:
        best_offsets, cal_f1 = calibrate_logit_offsets(logits, labels)
        title = "FINAL (best checkpoint, offsets fitted on the full dev set)"
    else:
        best_offsets = torch.zeros(len(LABELS))
        title = "FINAL (best checkpoint, no offsets)"
        print("offsets not used")

    preds = (logits + best_offsets).argmax(-1).tolist()
    final_f1 = print_scores(labels.tolist(), preds, domains=dev_domains, title=title)
    print("best uncalibrated dev macro-F1:", float(best_f1), "| final reported:", float(final_f1))

    return model, history, best_offsets, float(best_f1)


---

## Stage 2 complete

Everything above is shared infrastructure: four backbones load and smoke-test cleanly, the
tokenizer and data pipeline are built, `MAX_SENT_LEN` is set from the real distribution (not
assumed), and metrics/calibration/submission utilities are ready.

**Nothing experiment-specific is here yet.** Run this notebook up through this point on Kaggle,
confirm the smoke test prints look right (backbone param counts, `MAX_SENT_LEN`, class weight
table, label counts) and the data loads without errors, then tell me to continue -- I'll add
experiment #1 (GQA + LoRMA) next, followed by the rest in order.

---
# Stage 3: Experiments

## Experiment 1 -- GQA + LoRMA

**Technique:** LoRMA (Low-Rank Multiplicative Adaptation) -- the same from-scratch
implementation used in Phase 2, `W' = (I + (alpha/r) * B @ A) @ W0`, a rank-inflated
*multiplicative* update rather than LoRA's additive one. GQA backbone frozen entirely;
only the LoRMA B/A matrices and the new classification head are trainable.

**Targets:** `q_proj` and `v_proj` only (not `k_proj`, `c_proj`, or the MLP) -- the standard,
most commonly effective LoRA/LoRMA target set from the literature, and it sidesteps a real
implementation hazard: the MLP is an `nn.Sequential`, whose children are named `"0"`, `"1"`,
`"2"`, `"3"` -- numeric names that could collide with layer-index substrings elsewhere if
matched carelessly. Targeting the named attention projections avoids that risk entirely.

In [ ]:
class LoRMALinear(nn.Module):
    """LoRMA+ (rank-inflated multiplicative adaptation): W' = (I + (alpha/r) B@A) @ W0."""
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        BA = self.B @ self.A
        M = self.eye + (self.alpha / self.r) * BA
        W_eff = M @ self.base.weight
        return F.linear(x, W_eff, self.base.bias)


def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in model.named_modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model

print("LoRMA defined.")

In [ ]:
EXP1_NAME = "01_gqa_lorma"
paths = experiment_dirs(EXP1_NAME)

LORMA_R, LORMA_ALPHA = 8, 16
LORMA_TARGETS = ["q_proj", "v_proj"]

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_LORMA, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 5

backbone = load_backbone("gqa")
for p in backbone.parameters():
    p.requires_grad = False
backbone = apply_lorma(backbone, LORMA_TARGETS, r=LORMA_R, alpha=LORMA_ALPHA)

model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

lorma_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": lorma_params, "lr": LR_LORMA},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offsets, best_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

In [ ]:
run_info = {
    "experiment": EXP1_NAME, "phase1_model": "GQA", "technique": "LoRMA",
    "r": LORMA_R, "alpha": LORMA_ALPHA, "targets": LORMA_TARGETS,
    "lr_lorma": LR_LORMA, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)

prediction = write_submission(paths, model, offsets)

---
## Shared infra for experiments 2-8

Three small, additive things, built on top of your Stage 2/Experiment 1 code without changing
its behavior:

1. **`train_classifier` now also returns `final_f1`** (the possibly-calibrated score it already
   prints) alongside the uncalibrated `best_f1` it returned before -- needed so later experiments
   can programmatically find "whichever technique won" rather than you hardcoding it.
2. **Two registries**, filled in as each experiment trains: `FRESH_BACKBONE_BUILDERS[name]` returns
   an *untrained* backbone+adapter (used by #7 to retrain the winning technique on context-window
   data, starting from Phase 1 again, per the brief's rule); `EXPERIMENT_BUILDERS[name]` returns the
   already-*trained* model with its best checkpoint loaded (used by #6's teacher and #8's ensemble).
3. **A context-window dataset** for #7: previous 2 sentences + target sentence, truncated from the
   *left* if too long so the target sentence is never cut, target placed last so last-token pooling
   lands on it.

In [ ]:
def train_classifier(model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
                     class_weights, epochs, patience, paths, scaler=None, min_cal_gain=0.005):
    scaler = scaler or torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = {"train": [], "dev_f1": []}
    best_f1, bad_epochs = 0.0, 0
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(train_loader):
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**b, class_weights=class_weights)
            scaler.scale(out["loss"]).backward()
            run += out["loss"].item(); cnt += 1
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        logits, labels = get_logits(model, dev_loader)
        f1 = raw_macro_f1(logits, labels)
        history["train"].append(run / cnt); history["dev_f1"].append(f1)
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated) {f1:.4f} | {time.time()-t0:.0f}s")

        if f1 > best_f1:
            best_f1, bad_epochs = f1, 0
            torch.save(model.state_dict(), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
    logits, labels = get_logits(model, dev_loader)

    print("calibration check (fit on one half of the dev documents, test on the other half):")
    gain = calibration_check(logits, labels, [(r["domain"], r["doc_id"]) for r in dev_records])
    print(f"mean held-out gain from offsets: {gain:+.4f} (threshold {min_cal_gain})")
    if gain > min_cal_gain:
        best_offsets, cal_f1 = calibrate_logit_offsets(logits, labels)
        title = "FINAL (best checkpoint, offsets fitted on the full dev set)"
    else:
        best_offsets = torch.zeros(len(LABELS))
        title = "FINAL (best checkpoint, no offsets)"
        print("offsets not used")

    preds = (logits + best_offsets).argmax(-1).tolist()
    final_f1 = print_scores(labels.tolist(), preds, domains=dev_domains, title=title)
    print("best uncalibrated dev macro-F1:", float(best_f1), "| final reported:", float(final_f1))

    return model, history, best_offsets, float(best_f1), float(final_f1)   # <-- final_f1 added

print("train_classifier extended (now also returns final_f1).")

In [ ]:
import math

FRESH_BACKBONE_BUILDERS = {}   # name -> fn() -> untrained backbone+adapter
EXPERIMENT_BUILDERS = {}       # name -> fn() -> trained SentenceClassifier, weights loaded, eval()
EXPERIMENT_DATASET_TYPE = {}   # name -> "sentence" | "context"
EXPERIMENT_RUN_INFO = {}       # name -> run_info dict, kept in memory as each experiment finishes


def pick_best_experiment(candidate_names):
    """Highest recorded final_dev_f1 among the given experiment names."""
    scored = [(EXPERIMENT_RUN_INFO[n]["final_dev_f1"], n) for n in candidate_names if n in EXPERIMENT_RUN_INFO]
    assert scored, "none of the candidate experiments have recorded run_info yet"
    scored.sort(reverse=True)
    print("candidates (sorted by final dev macro-F1):")
    for f1, n in scored:
        print(f"  {n:30s} {f1:.4f}")
    return scored[0][1]

print("Registries defined.")

### Knowledge Distillation loss (for #6)

Standard hard-label + soft-target KD: cross-entropy against gold labels, plus a KL-divergence
term matching the student's softened logits to the (frozen) teacher's, at temperature `T`.

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, class_weights, T=2.0, alpha=0.5):
    hard = F.cross_entropy(student_logits, labels, weight=class_weights)
    soft = F.kl_div(F.log_softmax(student_logits / T, dim=-1),
                    F.softmax(teacher_logits / T, dim=-1), reduction="batchmean") * (T ** 2)
    return alpha * hard + (1 - alpha) * soft

print("distillation_loss defined.")

### Context-window dataset (for #7)

Builds, for each target sentence, previous-2-sentences + target, truncated from the left if the
combined window exceeds the token budget -- the target sentence itself is never truncated, and
since it's always last, last-token pooling in `SentenceClassifier` lands on it correctly without
any change to the model.

In [ ]:
CONTEXT_MAX_LEN = min(MAX_SENT_LEN * 3, MODEL_CONTEXT_LENGTH)
CONTEXT_WINDOW = 2   # previous N sentences

# group sentences by (domain, doc_id) in order, for fast previous-sentence lookup
_doc_sent_index = {}
for _recs in (train_records, dev_records, test_records):
    for r in _recs:
        _doc_sent_index.setdefault((r["domain"], r["doc_id"]), {})[r["sent_idx"]] = r["text"]


class ContextSentenceDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        doc_sents = _doc_sent_index[(r["domain"], r["doc_id"])]
        ctx_texts = [doc_sents[j] for j in range(max(0, r["sent_idx"] - CONTEXT_WINDOW), r["sent_idx"])]
        window_text = " ".join(ctx_texts + [r["text"]]) if ctx_texts else r["text"]
        ids = tokenizer.encode(window_text)
        if len(ids) > CONTEXT_MAX_LEN:
            ids = ids[-CONTEXT_MAX_LEN:]        # trim from the left -- keep the target's tail intact
        item = {"ids": ids, "idx": idx, "feats": r["feats"]}
        if self.with_labels:
            item["label"] = LABEL2ID[r["label"]]
        return item


def context_collator(batch):
    L = CONTEXT_MAX_LEN
    B = len(batch)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, -n:] = torch.tensor(item["ids"])     # right-align: pad on the LEFT
        attention_mask[i, -n:] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask,
           "record_idx": torch.tensor([item["idx"] for item in batch]),
           "feats": torch.tensor([item["feats"] for item in batch], dtype=torch.float)}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([item["label"] for item in batch])
    return out

_t = context_collator([ContextSentenceDataset(train_records)[i] for i in (0, 5, 50)])
print({k: v.shape for k, v in _t.items()})

**Note on left-padding here:** `SentenceClassifier`'s last-token pooling uses
`attention_mask.sum(dim=1) - 1`, which assumes the real tokens form a contiguous block and finds
the *last one*, regardless of whether padding sits on the left or right. Left-padding (above) is
therefore safe with the existing pooling code -- no changes needed to `SentenceClassifier` itself
for #7. But causal attention under left-padding needs the padding tokens masked out of the *keys*
(already handled: `attn_mask = causal & attention_mask[:, None, None, :].bool()`), which
`SentenceClassifier.forward` already does correctly.

---
## Experiment 2 -- GPT-2 + IA3

**Technique:** IA3 (Infused Adapter by Inhibiting and Amplifying Inner Activations) via
`peft.inject_adapter_in_model` -- your model is a plain `nn.Module`, not a `PreTrainedModel`, so
`get_peft_model` (which expects the HF wrapper conventions) isn't used; the lower-level injection
API attaches the adapter directly onto the named submodules instead.

**Targets:** `c_attn`, `c_proj`, `c_fc` (all three linears in GPT-2's attention+MLP), with `c_fc`
marked as the feedforward module (IA3 distinguishes attention-side scaling from feedforward-side
scaling, per the original paper). Base model fully frozen; only the injected IA3 vectors and the
new head are trainable -- printed explicitly below, per your requirement.

In [ ]:
from peft import IA3Config, inject_adapter_in_model

def build_backbone_gpt2_ia3():
    bb = load_backbone("gpt2")
    for p in bb.parameters():
        p.requires_grad = False
    ia3_config = IA3Config(target_modules=["c_attn", "c_proj", "c_fc"], feedforward_modules=["c_fc"])
    bb = inject_adapter_in_model(ia3_config, bb)
    return bb

FRESH_BACKBONE_BUILDERS["02_gpt2_ia3"] = build_backbone_gpt2_ia3
EXPERIMENT_DATASET_TYPE["02_gpt2_ia3"] = "sentence"
print("IA3 backbone builder registered.")

In [ ]:
EXP2_NAME = "02_gpt2_ia3"
paths = experiment_dirs(EXP2_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 5

backbone = FRESH_BACKBONE_BUILDERS[EXP2_NAME]()
model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

In [ ]:
run_info = {
    "experiment": EXP2_NAME, "phase1_model": "GPT-2", "technique": "IA3",
    "targets": ["c_attn", "c_proj", "c_fc"], "feedforward_modules": ["c_fc"],
    "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP2_NAME] = run_info

prediction = write_submission(paths, model, offsets)

_exp2_paths = paths
def build_exp2_trained():
    bb = build_backbone_gpt2_ia3()
    m = SentenceClassifier(bb, len(LABELS), N_EMBD, bidirectional=False).to(device)
    m.load_state_dict(torch.load(_exp2_paths["model_path"], weights_only=True))
    m.eval()
    return m
EXPERIMENT_BUILDERS[EXP2_NAME] = build_exp2_trained
print("registered:", EXP2_NAME)

---
## Experiment 3 -- SwiGLU model + LoRA

**Technique:** standard LoRA via `peft.inject_adapter_in_model`.

**Targets:** `w1`, `w3` -- SwiGLU's gate and up projections -- deliberately the MLP rather than
attention, unlike Experiment 1's attention-only targeting. SwiGLU's defining architectural feature
*is* its gated MLP, so adapting it there (rather than repeating "attention q/v" for a third time)
gives genuinely different, more informative evidence for the report's technique comparison.

In [ ]:
from peft import LoraConfig

def build_backbone_swiglu_lora():
    bb = load_backbone("swiglu")
    for p in bb.parameters():
        p.requires_grad = False
    lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["w1", "w3"], lora_dropout=0.05, bias="none")
    bb = inject_adapter_in_model(lora_config, bb)
    return bb

FRESH_BACKBONE_BUILDERS["03_swiglu_lora"] = build_backbone_swiglu_lora
EXPERIMENT_DATASET_TYPE["03_swiglu_lora"] = "sentence"
print("LoRA (SwiGLU) backbone builder registered.")

In [ ]:
EXP3_NAME = "03_swiglu_lora"
paths = experiment_dirs(EXP3_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 5

backbone = FRESH_BACKBONE_BUILDERS[EXP3_NAME]()
model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

In [ ]:
run_info = {
    "experiment": EXP3_NAME, "phase1_model": "SwiGLU", "technique": "LoRA",
    "r": 8, "alpha": 16, "targets": ["w1", "w3"],
    "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP3_NAME] = run_info

prediction = write_submission(paths, model, offsets)

_exp3_paths = paths
def build_exp3_trained():
    bb = build_backbone_swiglu_lora()
    m = SentenceClassifier(bb, len(LABELS), N_EMBD, bidirectional=False).to(device)
    m.load_state_dict(torch.load(_exp3_paths["model_path"], weights_only=True))
    m.eval()
    return m
EXPERIMENT_BUILDERS[EXP3_NAME] = build_exp3_trained
print("registered:", EXP3_NAME)

---
## Experiment 4 -- Prefix-LM + rsLoRA

**Technique:** Rank-Stabilized LoRA (`use_rslora=True`), which scales the LoRA update by
`alpha / sqrt(r)` instead of plain LoRA's `alpha / r` -- intended to keep the update's effective
magnitude more stable as rank increases, per the rsLoRA paper.

**Attention stays causal here**, matching Experiments 1-3, 5-6 -- per your correction, a
bidirectional mask only gets tested as a *separate* ablation for Prefix-LM (next cell after this
one), not as the main #4 submission, so #4 is directly comparable to the other techniques.

In [ ]:
def build_backbone_prefixlm_rslora():
    bb = load_backbone("prefixlm")
    for p in bb.parameters():
        p.requires_grad = False
    lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                             lora_dropout=0.05, bias="none", use_rslora=True)
    bb = inject_adapter_in_model(lora_config, bb)
    return bb

FRESH_BACKBONE_BUILDERS["04_prefixlm_rslora"] = build_backbone_prefixlm_rslora
EXPERIMENT_DATASET_TYPE["04_prefixlm_rslora"] = "sentence"
print("rsLoRA (Prefix-LM) backbone builder registered.")

In [ ]:
EXP4_NAME = "04_prefixlm_rslora"
paths = experiment_dirs(EXP4_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 5

backbone = FRESH_BACKBONE_BUILDERS[EXP4_NAME]()
model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

In [ ]:
run_info = {
    "experiment": EXP4_NAME, "phase1_model": "Prefix-LM", "technique": "rsLoRA",
    "r": 8, "alpha": 16, "targets": ["q_proj", "v_proj"], "use_rslora": True,
    "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1", "bidirectional": False,
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP4_NAME] = run_info

prediction = write_submission(paths, model, offsets)

_exp4_paths = paths
def build_exp4_trained():
    bb = build_backbone_prefixlm_rslora()
    m = SentenceClassifier(bb, len(LABELS), N_EMBD, bidirectional=False).to(device)
    m.load_state_dict(torch.load(_exp4_paths["model_path"], weights_only=True))
    m.eval()
    return m
EXPERIMENT_BUILDERS[EXP4_NAME] = build_exp4_trained
print("registered:", EXP4_NAME)

### Bonus ablation (not one of the 8) -- Prefix-LM + rsLoRA, bidirectional

Same technique and hyperparameters as #4, just with `bidirectional=True` -- the one architecture
actually pretrained with bidirectional-prefix attention, tested with it turned on for the
classification head too. Saved to its own folder so it never overwrites #4; useful evidence for
the viva ("why causal for everything else"), not counted toward the 8 required submissions.

In [ ]:
EXP4B_NAME = "04b_prefixlm_rslora_bidirectional_ablation"
paths = experiment_dirs(EXP4B_NAME)

backbone = build_backbone_prefixlm_rslora()
model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=True).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

run_info = {
    "experiment": EXP4B_NAME, "phase1_model": "Prefix-LM", "technique": "rsLoRA (bidirectional ablation)",
    "bidirectional": True, "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "note": "compare directly against 04_prefixlm_rslora (causal) -- not one of the 8 required submissions",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
print(f"\ncausal (exp 4) final dev F1: {EXPERIMENT_RUN_INFO['04_prefixlm_rslora']['final_dev_f1']:.4f}")
print(f"bidirectional (this ablation) final dev F1: {final_f1:.4f}")
prediction = write_submission(paths, model, offsets)

---
## Experiment 5 -- GQA + Prompt Tuning

**Technique:** hand-rolled in plain PyTorch -- `peft`'s `PromptTuningConfig`/`PromptEmbedding`
expect a Hugging Face `PreTrainedModel` with a `get_input_embeddings` convention and generate
loop that your custom GQA class doesn't have, so this is implemented directly: a block of
`n_prompt` learnable embedding vectors, prepended to the real token embeddings before the first
transformer block. The backbone is fully frozen; only the prompt vectors and the classification
head train.

Pooling happens over the **real-token region only** (the prompt positions are excluded from
last-token and mean pooling) -- the prompt is context for the backbone to condition on, not part
of the sentence representation itself.

In [ ]:
N_PROMPT = 20

class PromptTuningClassifier(nn.Module):
    def __init__(self, lm, n_labels, n_embd, n_prompt=N_PROMPT, dropout=0.1, n_feats=N_FEATS):
        super().__init__()
        self.lm = lm
        for p in self.lm.parameters():
            p.requires_grad = False
        self.n_prompt = n_prompt
        self.prompt = nn.Parameter(torch.randn(n_prompt, n_embd) * 0.02)
        self.n_feats = n_feats
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd + n_feats, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, n_labels)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None,
                logit_offsets=None, feats=None):
        B, T = input_ids.shape
        positions = torch.arange(self.n_prompt, self.n_prompt + T, device=input_ids.device)
        tok_x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        tok_x = self.lm.drop(tok_x)

        prompt = self.prompt.unsqueeze(0).expand(B, -1, -1)
        x = torch.cat([prompt, tok_x], dim=1)

        prompt_mask = torch.ones(B, self.n_prompt, dtype=torch.bool, device=input_ids.device)
        full_mask = torch.cat([prompt_mask, attention_mask], dim=1) if attention_mask is not None \
            else torch.ones(B, self.n_prompt + T, dtype=torch.bool, device=input_ids.device)
        Tfull = full_mask.size(1)
        causal = torch.tril(torch.ones(Tfull, Tfull, dtype=torch.bool, device=input_ids.device)).view(1, 1, Tfull, Tfull)
        attn_mask = causal & full_mask[:, None, None, :].bool()

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        real = x[:, self.n_prompt:, :]       # drop the prompt positions before pooling
        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            m = attention_mask.unsqueeze(-1).float()
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
            m = torch.ones(B, T, 1, device=input_ids.device)
        last = real[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((real.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(real.dtype)
        if feats is None:
            feats = torch.zeros(B, self.n_feats, device=input_ids.device)
        pooled = torch.cat([last, mean, feats.to(real.dtype)], dim=-1)

        logits = self.head(self.dropout(pooled))
        if logit_offsets is not None:
            logits = logits + logit_offsets
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        return {"loss": loss, "logits": logits}


def build_backbone_gqa_prompt():
    return load_backbone("gqa")   # frozen inside PromptTuningClassifier.__init__

FRESH_BACKBONE_BUILDERS["05_gqa_prompttuning"] = build_backbone_gqa_prompt
EXPERIMENT_DATASET_TYPE["05_gqa_prompttuning"] = "sentence"
print("Prompt Tuning classifier + builder defined.")

In [ ]:
EXP5_NAME = "05_gqa_prompttuning"
paths = experiment_dirs(EXP5_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_PROMPT, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-2, 1e-3, 0.05, 0.10, 5

backbone = FRESH_BACKBONE_BUILDERS[EXP5_NAME]()
model = PromptTuningClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, n_prompt=N_PROMPT).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

prompt_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": prompt_params, "lr": LR_PROMPT},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

In [ ]:
run_info = {
    "experiment": EXP5_NAME, "phase1_model": "GQA", "technique": "Prompt Tuning (hand-rolled)",
    "n_prompt": N_PROMPT, "lr_prompt": LR_PROMPT, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean (real tokens only)", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP5_NAME] = run_info

prediction = write_submission(paths, model, offsets)

_exp5_paths = paths
def build_exp5_trained():
    bb = build_backbone_gqa_prompt()
    m = PromptTuningClassifier(bb, len(LABELS), N_EMBD, n_prompt=N_PROMPT).to(device)
    m.load_state_dict(torch.load(_exp5_paths["model_path"], weights_only=True))
    m.eval()
    return m
EXPERIMENT_BUILDERS[EXP5_NAME] = build_exp5_trained
print("registered:", EXP5_NAME)

---
## Pick the winner of Experiments 1-5

Decides the teacher for #6 and the base technique for #7, from the actual recorded dev scores --
not assumed in advance.

In [ ]:
WINNER_CANDIDATES = ["01_gqa_lorma", "02_gpt2_ia3", "03_swiglu_lora", "04_prefixlm_rslora", "05_gqa_prompttuning"]
WINNER_NAME = pick_best_experiment(WINNER_CANDIDATES)
print("\nWinner:", WINNER_NAME)

---
## Experiment 6 -- Knowledge Distillation

**Teacher:** whichever of #1-#5 scored highest on dev (picked above), loaded frozen with its
trained checkpoint. **Student:** a smaller version of the *same Phase 1 architecture family* as
the teacher's backbone -- half the layers (6 of 12), initialized from the **first 6 of the
teacher's own pretrained Phase 1 layers** (not the teacher's fine-tuned adapter weights -- the
Phase 1 checkpoint itself), then trained fully (no adapter, every parameter trainable) against a
combined hard-label + soft-teacher-matching loss.

This is plain PyTorch, not a PEFT method -- per your rule, KD is written by hand.

In [ ]:
BACKBONE_OF = {
    "01_gqa_lorma": "gqa", "02_gpt2_ia3": "gpt2", "03_swiglu_lora": "swiglu",
    "04_prefixlm_rslora": "prefixlm", "05_gqa_prompttuning": "gqa",
}
STUDENT_N_LAYER = N_LAYER // 2

def build_student_backbone(winner_backbone_name):
    spec = BACKBONE_REGISTRY[winner_backbone_name]
    kwargs = dict(spec["kwargs"]); kwargs["n_layer"] = STUDENT_N_LAYER
    student = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **kwargs)

    full_state = load_file(os.path.join(MODEL_PATHS[winner_backbone_name], "model.safetensors"))
    student_state = student.state_dict()
    for k in list(student_state.keys()):
        if k.startswith("h."):
            layer_idx = int(k.split(".")[1])
            if layer_idx >= STUDENT_N_LAYER:
                continue   # shouldn't happen, student only has STUDENT_N_LAYER blocks
            src_key = k   # teacher's layer i -> student's layer i (first half)
        else:
            src_key = k
        if src_key in full_state and full_state[src_key].shape == student_state[k].shape:
            student_state[k] = full_state[src_key]
    student.load_state_dict(student_state)
    if hasattr(student, "lm_head"):
        del student.lm_head
    return student

teacher_backbone_name = BACKBONE_OF[WINNER_NAME]
print("teacher backbone family:", teacher_backbone_name, "| student layers:", STUDENT_N_LAYER, "of", N_LAYER)

In [ ]:
EXP6_NAME = "06_knowledge_distillation"
paths = experiment_dirs(EXP6_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 3e-5, 1e-3, 0.05, 0.10, 5
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.5

teacher_model = EXPERIMENT_BUILDERS[WINNER_NAME]()     # frozen, eval mode, trained checkpoint
for p in teacher_model.parameters():
    p.requires_grad = False
teacher_dataset_cls = ContextSentenceDataset if EXPERIMENT_DATASET_TYPE[WINNER_NAME] == "context" else SentenceDataset
teacher_collator = context_collator if EXPERIMENT_DATASET_TYPE[WINNER_NAME] == "context" else sentence_collator

student_backbone = build_student_backbone(teacher_backbone_name)
model = SentenceClassifier(student_backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)
for p in model.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
teacher_total = sum(p.numel() for p in teacher_model.parameters())
print(f"student trainable: {trainable:,} ({total:,} total) | teacher: {teacher_total:,}")

# teacher and student see the SAME input batches each step, but each may expect a different
# dataset type (sentence-only vs context-window) if the winner was experiment #7 -- not possible
# here since WINNER_NAME is chosen from #1-#5, all sentence-only, so one shared loader suffices.
train_loader = torch.utils.data.DataLoader(SentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=sentence_collator)
dev_loader = torch.utils.data.DataLoader(SentenceDataset(dev_records), batch_size=32,
                                         shuffle=False, collate_fn=sentence_collator)
dev_domains = [r["domain"] for r in dev_records]

backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
head_params = [p for n, p in model.named_parameters() if "head" in n]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history = {"train": [], "dev_f1": []}
best_f1, bad_epochs = 0.0, 0
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    run, cnt = 0.0, 0
    optimizer.zero_grad()
    for i, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            teacher_logits = teacher_model(input_ids=b["input_ids"], attention_mask=b["attention_mask"],
                                           feats=b.get("feats"))["logits"]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            student_out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], feats=b.get("feats"))
            loss = distillation_loss(student_out["logits"], teacher_logits, b["labels"], class_weights,
                                     T=KD_TEMPERATURE, alpha=KD_ALPHA)
        scaler.scale(loss).backward()
        run += loss.item(); cnt += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad()
        if scheduler is not None:
            scheduler.step()

    logits, labels = get_logits(model, dev_loader)
    f1 = raw_macro_f1(logits, labels)
    history["train"].append(run / cnt); history["dev_f1"].append(f1)
    print(f"epoch {epoch+1}/{EPOCHS} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated) {f1:.4f} | {time.time()-t0:.0f}s")

    if f1 > best_f1:
        best_f1, bad_epochs = f1, 0
        torch.save(model.state_dict(), paths["model_path"])
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
            break

model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
logits, labels = get_logits(model, dev_loader)
print("calibration check:")
gain = calibration_check(logits, labels, [(r["domain"], r["doc_id"]) for r in dev_records])
if gain > 0.005:
    offsets, _ = calibrate_logit_offsets(logits, labels)
else:
    offsets = torch.zeros(len(LABELS)); print("offsets not used")
preds = (logits + offsets).argmax(-1).tolist()
final_f1 = print_scores(labels.tolist(), preds, domains=dev_domains, title="FINAL (student, KD)")
print("best uncalibrated dev macro-F1:", best_f1, "| final reported:", final_f1)

teacher_final_f1 = EXPERIMENT_RUN_INFO[WINNER_NAME]["final_dev_f1"]
print(f"\nteacher ({WINNER_NAME}) final dev F1: {teacher_final_f1:.4f} | student (KD) final dev F1: {final_f1:.4f}")

In [ ]:
run_info = {
    "experiment": EXP6_NAME, "technique": "Knowledge Distillation",
    "teacher": WINNER_NAME, "teacher_backbone": teacher_backbone_name, "teacher_final_dev_f1": teacher_final_f1,
    "student_backbone": teacher_backbone_name, "student_n_layer": STUDENT_N_LAYER,
    "kd_temperature": KD_TEMPERATURE, "kd_alpha": KD_ALPHA,
    "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD, "max_sent_len": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "teacher_params": teacher_total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP6_NAME] = run_info
EXPERIMENT_DATASET_TYPE[EXP6_NAME] = "sentence"

prediction = write_submission(paths, model, offsets)

_exp6_paths = paths
_exp6_student_backbone_name = teacher_backbone_name
def build_exp6_trained():
    bb = build_student_backbone(_exp6_student_backbone_name)
    m = SentenceClassifier(bb, len(LABELS), N_EMBD, bidirectional=False).to(device)
    m.load_state_dict(torch.load(_exp6_paths["model_path"], weights_only=True))
    m.eval()
    return m
EXPERIMENT_BUILDERS[EXP6_NAME] = build_exp6_trained
print("registered:", EXP6_NAME)

---
## Experiment 7 -- Context-aware version of the winning technique

Retrains the **same technique and backbone that won #1-#5**, starting fresh from the Phase 1
checkpoint (per the brief's rule -- every experiment starts from Phase 1, not from an
already-fine-tuned model), using the context-window dataset from the shared-infra section
(previous 2 sentences + target, target pooled). Same hyperparameters as the winning experiment's
original run, so the only thing that changed is the input.

In [ ]:
EXP7_NAME = "07_context_aware_" + WINNER_NAME
paths = experiment_dirs(EXP7_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 16, 1
WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 0.05, 0.10, 5

backbone = FRESH_BACKBONE_BUILDERS[WINNER_NAME]()

if WINNER_NAME == "05_gqa_prompttuning":
    model = PromptTuningClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, n_prompt=N_PROMPT).to(device)
    adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
    LR_ADAPTER = 1e-2
else:
    model = SentenceClassifier(backbone, n_labels=len(LABELS), n_embd=N_EMBD, bidirectional=False).to(device)
    adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
    LR_ADAPTER = 1e-3

head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
LR_HEAD = 1e-3

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(ContextSentenceDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=context_collator)
dev_loader = torch.utils.data.DataLoader(ContextSentenceDataset(dev_records), batch_size=16,
                                         shuffle=False, collate_fn=context_collator)
dev_domains = [r["domain"] for r in dev_records]

optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps, "| context window:", CONTEXT_WINDOW, "sentences | max len:", CONTEXT_MAX_LEN)

In [ ]:
model, history, offsets, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
    class_weights, EPOCHS, PATIENCE, paths
)

winner_sentence_only_f1 = EXPERIMENT_RUN_INFO[WINNER_NAME]["final_dev_f1"]
print(f"\n{WINNER_NAME} sentence-only final dev F1: {winner_sentence_only_f1:.4f}")
print(f"{EXP7_NAME} context-aware final dev F1: {final_f1:.4f}")
print(f"context {'helped' if final_f1 > winner_sentence_only_f1 else 'did not help'} on this technique/backbone.")

In [ ]:
run_info = {
    "experiment": EXP7_NAME, "technique": "context-aware (prev 2 + target) + " + EXPERIMENT_RUN_INFO[WINNER_NAME]["technique"],
    "based_on": WINNER_NAME, "context_window": CONTEXT_WINDOW, "context_max_len": CONTEXT_MAX_LEN,
    "max_sent_len_sentence_only_baseline": MAX_SENT_LEN,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "sentence_only_final_dev_f1_for_comparison": winner_sentence_only_f1,
    "trainable_params": trainable, "total_params": total,
    "logit_offsets": offsets.tolist(),
    "pooling": "last+mean (target sentence region)", "features": "rel_pos, log_pos_from_start, log_pos_from_end, domain",
    "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP7_NAME] = run_info
EXPERIMENT_DATASET_TYPE[EXP7_NAME] = "context"

prediction = write_submission_context(paths, model, offsets) if False else None   # see next cell

`write_submission` (Stage 2) assumes `SentenceDataset`/`sentence_collator` -- #7 needs the
context-window version instead. Small variant, same structure:

In [ ]:
def write_submission_context(paths, model, offsets=None):
    test_ds = ContextSentenceDataset(test_records, with_labels=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=context_collator)

    model.eval()
    all_logits = []
    with torch.no_grad():
        for b in test_loader:
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], feats=b["feats"])
            all_logits.append(out["logits"].float().cpu())
    logits = torch.cat(all_logits)
    if offsets is not None:
        logits = logits + offsets.cpu()
    preds = logits.argmax(-1).tolist()

    by_doc = {}
    for rec, p in zip(test_records, preds):
        by_doc.setdefault(rec["doc_id"], {})[rec["sent_idx"]] = ID2LABEL[p]

    prediction, expected_ids = {}, set()
    for domain, docs in test_docs.items():
        items = docs.items() if isinstance(docs, dict) else [(d["id"], d["text"]) for d in docs]
        for doc_id, sents in items:
            expected_ids.add(doc_id)
            prediction[doc_id] = [by_doc[doc_id][i] for i in range(len(sents))]
    assert set(prediction) == expected_ids

    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    print("saved:", paths["submission_zip"])
    return prediction

prediction = write_submission_context(paths, model, offsets)

if WINNER_NAME == "05_gqa_prompttuning":
    def build_exp7_trained():
        bb = FRESH_BACKBONE_BUILDERS[WINNER_NAME]()
        m = PromptTuningClassifier(bb, len(LABELS), N_EMBD, n_prompt=N_PROMPT).to(device)
        m.load_state_dict(torch.load(paths["model_path"], weights_only=True))
        m.eval(); return m
else:
    def build_exp7_trained():
        bb = FRESH_BACKBONE_BUILDERS[WINNER_NAME]()
        m = SentenceClassifier(bb, len(LABELS), N_EMBD, bidirectional=False).to(device)
        m.load_state_dict(torch.load(paths["model_path"], weights_only=True))
        m.eval(); return m

EXPERIMENT_BUILDERS[EXP7_NAME] = build_exp7_trained
print("registered:", EXP7_NAME)

---
## Experiment 8 -- Ensemble

Averages **softmax class probabilities** (not raw logits -- different techniques/backbones can
sit at different logit scales, so probability-space averaging is the principled choice) across
all 7 prior saved checkpoints: #1-#6 on sentence-only input, #7 on its context-window input.
Calibration is re-run on the *ensembled* dev probabilities (converted back to log-space) --
fitting offsets per-model and then averaging the offsets would not be equivalent.

In [ ]:
ENSEMBLE_MEMBERS = ["01_gqa_lorma", "02_gpt2_ia3", "03_swiglu_lora", "04_prefixlm_rslora",
                    "05_gqa_prompttuning", EXP6_NAME, EXP7_NAME]
print("ensemble members:", ENSEMBLE_MEMBERS)

@torch.no_grad()
def get_probs(model, records, dataset_type):
    if dataset_type == "context":
        ds = ContextSentenceDataset(records, with_labels=("label" in records[0]))
        loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=False, collate_fn=context_collator)
    else:
        ds = SentenceDataset(records, with_labels=("label" in records[0]))
        loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False, collate_fn=sentence_collator)
    model.eval()
    all_probs, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], feats=b.get("feats"))
        all_probs.append(F.softmax(out["logits"].float(), dim=-1).cpu())
        if "labels" in b:
            all_labels.append(b["labels"].cpu())
    probs = torch.cat(all_probs)
    labels = torch.cat(all_labels) if all_labels else None
    return probs, labels


dev_prob_sum, dev_labels = None, None
for name in ENSEMBLE_MEMBERS:
    m = EXPERIMENT_BUILDERS[name]()
    p, labels = get_probs(m, dev_records, EXPERIMENT_DATASET_TYPE.get(name, "sentence"))
    dev_prob_sum = p if dev_prob_sum is None else dev_prob_sum + p
    dev_labels = labels
    del m
    torch.cuda.empty_cache()
    print(f"  {name}: done")

dev_probs_avg = dev_prob_sum / len(ENSEMBLE_MEMBERS)
EPS = 1e-9
dev_logprobs_avg = torch.log(dev_probs_avg + EPS)   # treat as logits for the existing calibration utilities

In [ ]:
EXP8_NAME = "08_ensemble"
paths = experiment_dirs(EXP8_NAME)
dev_domains = [r["domain"] for r in dev_records]

f1_uncalibrated = raw_macro_f1(dev_logprobs_avg, dev_labels)
print(f"ensemble dev macro-F1 (uncalibrated): {f1_uncalibrated:.4f}")

print("calibration check (fit on one half of the dev documents, test on the other half):")
gain = calibration_check(dev_logprobs_avg, dev_labels, [(r["domain"], r["doc_id"]) for r in dev_records])
print(f"mean held-out gain from offsets: {gain:+.4f}")
if gain > 0.005:
    offsets, _ = calibrate_logit_offsets(dev_logprobs_avg, dev_labels)
else:
    offsets = torch.zeros(len(LABELS)); print("offsets not used")

preds = (dev_logprobs_avg + offsets).argmax(-1).tolist()
final_f1 = print_scores(dev_labels.tolist(), preds, domains=dev_domains, title="FINAL (ensemble, calibrated)")

print("\nindividual members vs. ensemble:")
for name in ENSEMBLE_MEMBERS:
    print(f"  {name:35s} final_dev_f1={EXPERIMENT_RUN_INFO[name]['final_dev_f1']:.4f}")
print(f"  {'ENSEMBLE':35s} final_dev_f1={final_f1:.4f}")

In [ ]:
test_prob_sum = None
for name in ENSEMBLE_MEMBERS:
    m = EXPERIMENT_BUILDERS[name]()
    p, _ = get_probs(m, test_records, EXPERIMENT_DATASET_TYPE.get(name, "sentence"))
    test_prob_sum = p if test_prob_sum is None else test_prob_sum + p
    del m
    torch.cuda.empty_cache()

test_probs_avg = test_prob_sum / len(ENSEMBLE_MEMBERS)
test_logprobs_avg = torch.log(test_probs_avg + EPS) + offsets
preds = test_logprobs_avg.argmax(-1).tolist()

by_doc = {}
for rec, p in zip(test_records, preds):
    by_doc.setdefault(rec["doc_id"], {})[rec["sent_idx"]] = ID2LABEL[p]

prediction, expected_ids = {}, set()
for domain, docs in test_docs.items():
    items = docs.items() if isinstance(docs, dict) else [(d["id"], d["text"]) for d in docs]
    for doc_id, sents in items:
        expected_ids.add(doc_id)
        prediction[doc_id] = [by_doc[doc_id][i] for i in range(len(sents))]
assert set(prediction) == expected_ids
for doc_id, labs in prediction.items():
    assert all(l in LABELS for l in labs), doc_id

with open(paths["prediction_path"], "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False)
with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(paths["prediction_path"], arcname="prediction.json")
print("saved:", paths["submission_zip"])
with zipfile.ZipFile(paths["submission_zip"]) as z:
    print(z.namelist())

run_info = {
    "experiment": EXP8_NAME, "technique": "Ensemble (averaged softmax probabilities)",
    "members": ENSEMBLE_MEMBERS, "n_members": len(ENSEMBLE_MEMBERS),
    "best_dev_f1": f1_uncalibrated, "final_dev_f1": final_f1,
    "logit_offsets": offsets.tolist(),
    "selection_metric": "uncalibrated ensembled dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
EXPERIMENT_RUN_INFO[EXP8_NAME] = run_info

---
## All 8 submissions -- summary

Run this last, once every experiment above has finished, for a single table to paste into the
report.

In [ ]:
import glob

print(f"{'experiment':35s} {'final_dev_f1':>13s}")
for d in sorted(glob.glob(os.path.join(BASE_OUTPUT_DIR, "*", "run_info.json"))):
    with open(d) as f:
        ri = json.load(f)
    f1 = ri.get("final_dev_f1", ri.get("best_dev_f1", float("nan")))
    print(f"{ri['experiment']:35s} {f1:13.4f}")